# MOESCORE Full10 — Kaggle Save & Run All

Pipeline tự động kế thừa cách chạy đã kiểm chứng của Full9, bổ sung M2D-LoRA CORAL làm expert thứ 10. Gắn các Kaggle Input riêng tư cần thiết, bật Internet để clone/cài dependencies và chọn hai GPU T4 trước khi **Save Version → Save & Run All**. Không tải checkpoint Expert 10 từ Internet và không tự upload cache.

Cell cấu hình đầu tiên là nơi duy nhất cần chỉnh. Nếu có cache Full9, khai báo thư mục Input ở đó; nếu không, pipeline sẽ extraction đủ mười expert.


In [ ]:
from pathlib import Path

# Kaggle Inputs đã được xác nhận trong notebook Full9 thành công.
DATASET_ROOT = Path(
    "/kaggle/input/datasets/qnv8071/xacle-2027-data/"
    "XACLE_Challenge_2027_development_dataset"
)
WEIGHTS_DATASET_ROOT = Path(
    "/kaggle/input/datasets/qnv8071/xacle-weights"
)
WEIGHTS_ROOT = (
    WEIGHTS_DATASET_ROOT / "weights" / "weights"
)
HF_CACHE_DATASET_ROOT = Path(
    "/kaggle/input/datasets/qnv8071/moescore-hf-text-cache-v2"
)
HF_CACHE_ARCHIVE = None  # Tùy chọn: đường dẫn chính xác tới hf_hub_cache.tar
HF_CACHE_MANIFEST = None  # Tùy chọn: manifest.json kề archive

EXPERT_PROFILE = "full_10"
REPO_URL = "https://github.com/vietquang8071/MOESCORE.git"
REPO_REF = "feature/moe-10-m2d-lora-coral"
# Pin tới commit chứa implementation MOE-10.2; SHA không lấy từ commit notebook pin-only này.
REPO_COMMIT_SHA = "867cc84d00d0cf00e53aed183e04f69f9287948e"
PROJECT_ROOT = Path("/kaggle/working/MOESCORE")

KAGGLE_USERNAME = "qnv8071"
FEATURE_DATASET_SLUG = "moescore-full10-feature-cache"
FEATURE_DATASET_HANDLE = f"{KAGGLE_USERNAME}/{FEATURE_DATASET_SLUG}"

# Để None thì tìm đúng một file tên m2d-lora-Nam.pt dưới /kaggle/input.
# Nếu có nhiều checkpoint cùng tên, đặt đường dẫn tuyệt đối tới file cần dùng.
EXPERT10_CHECKPOINT = None
# Có thể đặt `Path("/kaggle/input/YOUR_EXPERT10_DATASET/m2d-lora-Nam.pt")`
# nếu muốn chọn chính xác một Kaggle Input thay vì tự động tìm.
EXPERT10_EXPECTED_SHA256 = "d6edbec6b724be48a18f618c98a2e25d044f06108e8715d86b58c59e97e6cbdb"
CHECKPOINT_OVERRIDES = {
    "top2_msclap.checkpoint": str(
        WEIGHTS_DATASET_ROOT / "CLAP_weights_2023.pth"
    ),
    "m2d_lora.finetuned_checkpoint": str(
        WEIGHTS_DATASET_ROOT / "m2d-lora.pt"
    ),
}

# Có thể là Kaggle Input đã upload dạng archive hoặc thư mục feature_cache/.
# None là hợp lệ: khi đó expert cache trống và extraction đủ 10 expert sẽ chạy.
EXISTING_FEATURE_CACHE = None
GATE_RESUME_CHECKPOINT = None  # Chỉ dùng checkpoint Full10 có provenance khớp.

WORK_ROOT = Path("/kaggle/working/moescore")
SMOKE_CACHE_ROOT = WORK_ROOT / "smoke"
FEATURE_CACHE_ROOT = WORK_ROOT / "extraction"
RUNS_ROOT = WORK_ROOT / "runs"
CONFIG_ROOT = WORK_ROOT / "configs"
LOG_ROOT = WORK_ROOT / "logs"
HF_HOME = WORK_ROOT / "huggingface"
UPLOAD_DIRECTORY = WORK_ROOT / "feature-cache-upload"
UNILM_SOURCE_DIR = None  # Dùng source UniLM đã kiểm chứng trong repository.

FULL10_ORDER = (
    "expert1", "expert2", "expert3", "expert4", "top2_m2d",
    "top2_msclap", "top2_laion", "top2_mga", "m2d_lora", "m2d_lora_coral",
)
WORKER_DEVICES = {
    "expert1": "cuda:0",
    "expert2": "cuda:1",
    "expert3": "cuda:0",
    "expert4": "cuda:1",
    "top2_m2d": "cuda:0",
    "top2_msclap": "cuda:1",
    "top2_laion": "cuda:0",
    "top2_mga": "cuda:1",
    "m2d_lora": "cuda:0",
    "m2d_lora_coral": "cuda:1",
}
EXTRACTION_BATCH_SIZES = {
    name: (1 if name == "m2d_lora_coral" else 4)
    for name in WORKER_DEVICES
}
SHARD_SIZE = 128
SMOKE_SAMPLES_PER_SPLIT = 4

# Save & Run All defaults: bounded smoke, full extraction, independent Full10 gate,
# validation, và dự đoán test không nhãn.
SMOKE_ONLY = False
RUN_GPU_SMOKE = True
RUN_FULL_EXTRACTION = True
RUN_GATE_TRAINING = True
RUN_EVALUATION = True
RUN_TEST_INFERENCE = True
RUN_TEST_EVALUATION = False
UPLOAD_FEATURE_CACHE = False
REBUILD_HF_CACHE = False
UPLOAD_REPAIRED_HF_CACHE = False
HF_CACHE_UPLOAD_HANDLE = None  # Chỉ maintenance mode; dataset upload phải private.

INSTALL_MISSING_DEPENDENCIES = True
WHEELHOUSE = None  # Thư mục wheel offline tùy chọn.
GATE_DEVICE = "cuda:0"
TRAIN_BATCH_SIZE = 8
TRAIN_EPOCHS = 50
TRAIN_LEARNING_RATE = 1e-5
SEED = 2027

RUN_STAGES = (
    "runtime_environment", "source_checkout", "dependency_setup", "input_discovery",
    "hf_cache_restore", "config_generation", "offline_preflight", "gpu_smoke",
    "smoke_gate", "existing_feature_cache_restore", "full_extraction",
    "feature_store_validation", "feature_cache_upload", "gate_training",
    "validation_evaluation", "test_inference", "test_evaluation",
    "final_artifact_verification",
)
assert EXPERT_PROFILE == "full_10" and SMOKE_SAMPLES_PER_SPLIT == 4


In [ ]:
"""Small, testable safety helpers used by the Full10 Kaggle notebook.

The helpers deliberately do not import MOESCORE experts or initialize CUDA.
Extraction, cache identity checks, feature-store assembly, training, and
evaluation remain in the existing MOESCORE scripts.
"""
from __future__ import annotations

# MOE10_KAGGLE_SUPPORT

from contextlib import contextmanager
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import re
import shutil
import subprocess
import tarfile
import tempfile
import time
from typing import Any, Iterable, Mapping


EXPERT10_CHECKPOINT_NAME = "m2d-lora-Nam.pt"
EXPERT10_CHECKPOINT_SHA256 = "d6edbec6b724be48a18f618c98a2e25d044f06108e8715d86b58c59e97e6cbdb"
HF_REQUIRED_MODELS = (
    "roberta-base",
    "bert-base-uncased",
    "google-bert/bert-base-uncased",
    "gpt2",
    "facebook/bart-base",
)
_SHA256_RE = re.compile(r"^[a-fA-F0-9]{64}$")
_GIT_SHA_RE = re.compile(r"^(?:[a-fA-F0-9]{40}|[a-fA-F0-9]{64})$")


def sha256_file(path: str | os.PathLike[str]) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_expert10_checkpoint(
    input_root: str | os.PathLike[str],
    *,
    configured_path: str | os.PathLike[str] | None = None,
    expected_sha256: str = EXPERT10_CHECKPOINT_SHA256,
    filename: str = EXPERT10_CHECKPOINT_NAME,
) -> Path:
    """Resolve an explicitly selected E10 file or one unique exact-name input."""
    if not _SHA256_RE.fullmatch(expected_sha256):
        raise ValueError("Expert 10 expected SHA-256 must be 64 hexadecimal characters")
    if configured_path is not None:
        selected = Path(configured_path).expanduser().resolve()
        if not selected.is_file():
            raise FileNotFoundError(
                f"Configured Expert 10 checkpoint is missing: {selected}; attach the private dataset containing {filename}."
            )
    else:
        root = Path(input_root).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(
                f"Kaggle input root is missing: {root}; attach the private dataset containing {filename}."
            )
        matches = sorted(path for path in root.rglob(filename) if path.is_file() and path.name == filename)
        if not matches:
            raise FileNotFoundError(
                f"No {filename} found below {root}; attach the private Expert 10 Kaggle Dataset or set EXPERT10_CHECKPOINT explicitly."
            )
        if len(matches) != 1:
            raise ValueError(
                f"Found multiple {filename} checkpoints: {matches}; set EXPERT10_CHECKPOINT to exactly one intended file."
            )
        selected = matches[0].resolve()
    observed_sha = sha256_file(selected)
    if observed_sha.lower() != expected_sha256.lower():
        raise ValueError(
            f"Expert 10 checkpoint SHA-256 mismatch for {selected}: expected {expected_sha256}, got {observed_sha}."
        )
    return selected


def resolve_hf_cache_inputs(
    input_root: str | os.PathLike[str],
    *,
    dataset_root: str | os.PathLike[str] | None = None,
    archive_path: str | os.PathLike[str] | None = None,
    manifest_path: str | os.PathLike[str] | None = None,
) -> tuple[Path, Path]:
    """Locate one HF archive and its adjacent manifest without treating it as a Hub directory."""
    if archive_path is not None:
        archive = Path(archive_path).expanduser().resolve()
        if not archive.is_file() or archive.name != "hf_hub_cache.tar":
            raise FileNotFoundError(f"HF cache archive must be the attached hf_hub_cache.tar file: {archive}")
    else:
        root = Path(dataset_root).expanduser().resolve() if dataset_root is not None else Path(input_root).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(f"HF cache Kaggle Input directory is missing: {root}")
        matches = sorted(path for path in root.rglob("hf_hub_cache.tar") if path.is_file())
        if not matches:
            raise FileNotFoundError(
                f"No hf_hub_cache.tar found below {root}; attach qnv8071/moescore-hf-text-cache-v2."
            )
        if len(matches) != 1:
            raise ValueError(f"Found multiple hf_hub_cache.tar archives below {root}: {matches}; select one explicitly.")
        archive = matches[0].resolve()
    if manifest_path is not None:
        manifest = Path(manifest_path).expanduser().resolve()
    else:
        manifest = archive.parent / "manifest.json"
    if not manifest.is_file():
        raise FileNotFoundError(f"HF cache manifest is missing beside the archive: {manifest}")
    return archive, manifest


def _validate_hf_manifest(
    archive: Path,
    manifest_path: Path,
    required_models: Iterable[str],
) -> dict[str, Any]:
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        raise ValueError(f"Cannot read HF cache manifest {manifest_path}: {exc}") from exc
    if not isinstance(manifest, dict):
        raise ValueError("HF cache manifest must contain a JSON object")
    archive_name = manifest.get("archive_name", manifest.get("archive"))
    archive_sha = manifest.get("archive_sha256", manifest.get("sha256"))
    if archive_name != "hf_hub_cache.tar" or archive.name != archive_name:
        raise ValueError("HF cache manifest must identify the attached hf_hub_cache.tar")
    if not isinstance(archive_sha, str) or not _SHA256_RE.fullmatch(archive_sha):
        raise ValueError("HF cache manifest has an invalid SHA-256")
    listed_models = manifest.get("models")
    if not isinstance(listed_models, list) or any(not isinstance(item, str) for item in listed_models):
        raise ValueError("HF cache manifest must list model resources")
    if len(listed_models) != len(set(listed_models)):
        raise ValueError("HF cache manifest lists duplicate model resources")
    missing = sorted(set(required_models) - set(listed_models))
    if missing:
        raise ValueError(f"HF cache manifest is missing required model resources: {missing}")
    if manifest.get("offline_preflight") not in {None, "PASS"}:
        raise ValueError("HF cache manifest does not record a successful offline preflight")
    if "archive_size_bytes" in manifest:
        size = manifest["archive_size_bytes"]
        if type(size) is not int or size != archive.stat().st_size:
            raise ValueError("HF cache archive size differs from its manifest")
    observed_sha = sha256_file(archive)
    if observed_sha.lower() != archive_sha.lower():
        raise ValueError(f"HF cache archive SHA-256 mismatch: expected {archive_sha}, got {observed_sha}")
    return {**manifest, "archive_sha256": observed_sha, "models": listed_models}


def _safe_member_path(name: str) -> PurePosixPath:
    path = PurePosixPath(name)
    if path.is_absolute() or not path.parts or any(part in {"", ".", ".."} for part in path.parts):
        raise ValueError(f"Unsafe HF cache archive member path: {name!r}")
    if path.parts[0] != "hub":
        raise ValueError(f"HF cache archive member must be rooted under hub/: {name!r}")
    return path


def _normalized_relative_target(member_path: PurePosixPath, linkname: str) -> PurePosixPath:
    target = PurePosixPath(linkname)
    if target.is_absolute() or not target.parts:
        raise ValueError(f"HF cache symlink must have a non-empty relative target: {member_path} -> {linkname!r}")
    stack = list(member_path.parent.parts)
    for part in target.parts:
        if part in {"", "."}:
            continue
        if part == "..":
            if len(stack) <= 1:  # Keep the target within the hub/ tree.
                raise ValueError(f"HF cache symlink escapes hub/: {member_path} -> {linkname!r}")
            stack.pop()
        else:
            stack.append(part)
    normalized = PurePosixPath(*stack)
    if not normalized.parts or normalized.parts[0] != "hub":
        raise ValueError(f"HF cache symlink escapes hub/: {member_path} -> {linkname!r}")
    return normalized


def _validate_hf_tree(hub: Path, required_models: Iterable[str]) -> tuple[int, int]:
    if not hub.is_dir():
        raise ValueError(f"Restored Hugging Face Hub directory is missing: {hub}")
    symlinks = 0
    for model_id in required_models:
        model_root = hub / ("models--" + model_id.replace("/", "--"))
        snapshots = model_root / "snapshots"
        if not model_root.is_dir() or not snapshots.is_dir() or not any(p.is_dir() for p in snapshots.iterdir()):
            raise ValueError(f"HF cache is missing a snapshot tree for {model_id}")
    for path in hub.rglob("*"):
        if path.is_symlink():
            symlinks += 1
            try:
                resolved = path.resolve(strict=True)
                resolved.relative_to(hub.resolve())
            except (OSError, RuntimeError, ValueError) as exc:
                raise ValueError(f"HF cache has a broken or escaping symlink: {path} -> {os.readlink(path)}") from exc
        elif path.is_file() and path.stat().st_size == 0:
            raise ValueError(f"HF cache contains an empty file: {path}")
    return sum(1 for p in hub.rglob("*") if p.is_file() or p.is_symlink()), symlinks


def restore_hf_cache(
    archive_path: str | os.PathLike[str],
    manifest_path: str | os.PathLike[str],
    hf_home: str | os.PathLike[str],
    *,
    required_models: Iterable[str] = HF_REQUIRED_MODELS,
) -> dict[str, Any]:
    """Verify and restore the HF Hub tar, rejecting traversal and unsafe symlinks."""
    archive = Path(archive_path).expanduser().resolve()
    manifest_file = Path(manifest_path).expanduser().resolve()
    target_home = Path(hf_home).expanduser().resolve()
    required_models = tuple(required_models)
    manifest = _validate_hf_manifest(archive, manifest_file, required_models)
    target_home.mkdir(parents=True, exist_ok=True)
    hub = target_home / "hub"
    if hub.exists():
        files, links = _validate_hf_tree(hub, required_models)
        return {
            "status": "REUSED_EXISTING",
            "hub_cache": str(hub),
            "archive_sha256": manifest["archive_sha256"],
            "models": list(required_models),
            "file_count": files,
            "symlink_count": links,
        }

    with tempfile.TemporaryDirectory(prefix="moescore-hf-restore-", dir=target_home.parent) as temporary:
        temporary_root = Path(temporary)
        extracted_hub = temporary_root / "hub"
        extracted_hub.mkdir()
        with tarfile.open(archive, mode="r:") as tar:
            members = tar.getmembers()
            checked: list[tuple[tarfile.TarInfo, PurePosixPath]] = []
            member_paths: set[PurePosixPath] = set()
            symlink_paths: set[PurePosixPath] = set()
            for member in members:
                relative = _safe_member_path(member.name)
                if relative in member_paths:
                    raise ValueError(f"Duplicate HF cache archive member: {member.name}")
                member_paths.add(relative)
                if member.isdir() or member.isfile():
                    pass
                elif member.issym():
                    _normalized_relative_target(relative, member.linkname)
                    symlink_paths.add(relative)
                else:
                    raise ValueError(f"Unsupported or unsafe HF cache archive entry: {member.name}")
                checked.append((member, relative))
            for _, relative in checked:
                if any(parent in symlink_paths for parent in relative.parents):
                    raise ValueError(f"HF archive writes below a symlink path: {relative}")

            # Create directories first, then regular files, then relative symlinks.
            directories = [(m, p) for m, p in checked if m.isdir()]
            regular_files = [(m, p) for m, p in checked if m.isfile()]
            symlink_members = [(m, p) for m, p in checked if m.issym()]
            for member, relative in sorted(directories, key=lambda item: len(item[1].parts)):
                destination = temporary_root.joinpath(*relative.parts)
                destination.mkdir(parents=True, exist_ok=True)
            for member, relative in regular_files:
                destination = temporary_root.joinpath(*relative.parts)
                destination.parent.mkdir(parents=True, exist_ok=True)
                if any(parent.is_symlink() for parent in destination.parents if parent != temporary_root.parent):
                    raise ValueError(f"HF archive destination traverses a symlink: {relative}")
                source = tar.extractfile(member)
                if source is None:
                    raise ValueError(f"Cannot read HF archive member: {member.name}")
                with source, destination.open("xb") as output:
                    shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
            for member, relative in symlink_members:
                destination = temporary_root.joinpath(*relative.parts)
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.symlink_to(member.linkname)

        _, links = _validate_hf_tree(extracted_hub, required_models)
        if hub.exists():
            raise FileExistsError(f"Refusing to replace an existing HF Hub cache: {hub}")
        os.replace(extracted_hub, hub)
        files, links = _validate_hf_tree(hub, required_models)
    return {
        "status": "PASS",
        "hub_cache": str(hub),
        "archive_sha256": manifest["archive_sha256"],
        "models": list(required_models),
        "file_count": files,
        "symlink_count": links,
    }


def ensure_source_checkout(
    repository_url: str,
    repository_ref: str,
    commit_sha: str,
    project_root: str | os.PathLike[str],
) -> str:
    """Clone the configured ref and detach at an exact commit; never replace stale source."""
    if not _GIT_SHA_RE.fullmatch(commit_sha):
        raise ValueError("REPO_COMMIT_SHA must be a full 40- or 64-character hexadecimal Git SHA")
    if not repository_url or not repository_ref:
        raise ValueError("Both REPO_URL and REPO_REF must be configured")
    root = Path(project_root).expanduser().resolve()

    def git(*args: str) -> str:
        result = subprocess.run(
            ["git", "-C", str(root), *args],
            check=True,
            capture_output=True,
            text=True,
        )
        return result.stdout.strip()

    if root.exists():
        if not (root / ".git").exists():
            raise ValueError(
                f"Project path already exists without Git metadata: {root}; choose a fresh PROJECT_ROOT."
            )
        try:
            actual = git("rev-parse", "HEAD")
        except subprocess.CalledProcessError as exc:
            raise ValueError(f"Cannot inspect existing MOESCORE checkout at {root}: {exc.stderr}") from exc
        if actual.lower() != commit_sha.lower():
            raise ValueError(
                f"Stale checkout at {root}: expected commit {commit_sha}, found {actual}. "
                "Choose a new PROJECT_ROOT; the notebook will not delete or silently reuse it."
            )
        return actual

    root.parent.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run(
            ["git", "clone", "--branch", repository_ref, "--single-branch", repository_url, str(root)],
            check=True,
            capture_output=True,
            text=True,
        )
        subprocess.run(
            ["git", "-C", str(root), "checkout", "--detach", commit_sha],
            check=True,
            capture_output=True,
            text=True,
        )
        actual = git("rev-parse", "HEAD")
    except subprocess.CalledProcessError as exc:
        # Only the directory created by this failed clone attempt is removed.
        if root.exists():
            shutil.rmtree(root)
        raise RuntimeError(
            f"Cannot create the pinned MOESCORE checkout for ref={repository_ref!r}, SHA={commit_sha}: "
            f"{exc.stderr or exc.stdout}. Push/attach the commit or correct REPO_REF/REPO_COMMIT_SHA."
        ) from exc
    if actual.lower() != commit_sha.lower():
        raise RuntimeError(f"Git checkout resolved to {actual}, expected {commit_sha}")
    return actual


def validate_source_files(project_root: str | os.PathLike[str]) -> list[str]:
    root = Path(project_root).resolve()
    required = (
        "model/expert_adapters/m2d_lora_coral_adapter.py",
        "model/expert_adapters/m2d_lora_coral_runtime.py",
        "model/expert_adapters/registry.py",
        "model/moe_gating_core.py",
        "configs/moe_full10.example.yaml",
        "data/moe_feature_store.py",
        "data/moe_training_dataset.py",
        "training/checkpoint_manager.py",
        "training/config.py",
        "training/moe_evaluator.py",
        "scripts/kaggle_full9_dual_gpu.py",
        "scripts/kaggle_full9_artifacts.py",
        "scripts/kaggle_moe_pipeline.py",
        "scripts/extract_moe_feature_store.py",
        "scripts/prepare_moe_text_cache.py",
        "scripts/train_moe_gating.py",
        "scripts/evaluate_moe_gating.py",
        "scripts/infer_moe_gating.py",
        "requirements-kaggle-py312.txt",
        "requirements-kaggle-py313.txt",
    )
    missing = [name for name in required if not (root / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Pinned source is missing required Full10 files: {missing}")
    return list(required)


def cache_shard_statistics(runtime: Mapping[str, Any], expected_shards: int) -> dict[str, int]:
    if type(expected_shards) is not int or expected_shards < 0:
        raise ValueError("expected_shards must be a non-negative integer")
    reused = runtime.get("reused_validated_shards", 0)
    if type(reused) is not int or not 0 <= reused <= expected_shards:
        raise ValueError("worker reused_validated_shards is missing, invalid, or exceeds expected shards")
    return {
        "expected_shards": expected_shards,
        "reused_shards": reused,
        "extracted_shards": expected_shards - reused,
    }


def rebuild_hf_cache(
    hf_home: str | os.PathLike[str],
    *,
    upload: bool = False,
    dataset_handle: str | None = None,
    project_root: str | os.PathLike[str] | None = None,
) -> dict[str, Any]:
    """Explicit maintenance-only rebuild. The default notebook path never calls this."""
    from huggingface_hub import snapshot_download

    home = Path(hf_home).expanduser().resolve()
    hub = home / "hub"
    hub.mkdir(parents=True, exist_ok=True)
    online_env = os.environ.copy()
    online_env.update({"HF_HOME": str(home), "HF_HUB_CACHE": str(hub), "HF_HUB_OFFLINE": "0", "TRANSFORMERS_OFFLINE": "0"})
    os.environ.update({key: online_env[key] for key in ("HF_HOME", "HF_HUB_CACHE", "HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE")})
    import huggingface_hub.constants as hub_constants
    hub_constants.HF_HUB_OFFLINE = False
    for model_id in HF_REQUIRED_MODELS:
        snapshot_download(
            repo_id=model_id,
            cache_dir=str(hub),
            allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors", "pytorch_model.bin"],
            local_files_only=False,
            force_download=True,
            max_workers=4,
        )
    file_count, symlink_count = _validate_hf_tree(hub, HF_REQUIRED_MODELS)
    result: dict[str, Any] = {
        "status": "PASS",
        "hub_cache": str(hub),
        "models": list(HF_REQUIRED_MODELS),
        "file_count": file_count,
        "symlink_count": symlink_count,
        "uploaded": False,
    }
    if upload:
        if not dataset_handle or not project_root:
            raise ValueError("HF cache upload requires an explicit private dataset handle and project root")
        from scripts.kaggle_full9_artifacts import check_private_upload_target
        check_private_upload_target(dataset_handle)
        import kagglehub
        with tempfile.TemporaryDirectory(prefix="moescore-hf-maintenance-upload-") as temporary:
            upload_dir = Path(temporary)
            archive_path = upload_dir / "hf_hub_cache.tar"
            with tarfile.open(archive_path, mode="w", dereference=False) as archive:
                for model_id in HF_REQUIRED_MODELS:
                    folder = hub / ("models--" + model_id.replace("/", "--"))
                    archive.add(folder, arcname=f"hub/{folder.name}", recursive=True)
            manifest = {
                "archive": archive_path.name,
                "sha256": sha256_file(archive_path),
                "models": list(HF_REQUIRED_MODELS),
                "offline_preflight": "PASS",
            }
            (upload_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
            kagglehub.dataset_upload(
                handle=dataset_handle,
                local_dataset_dir=str(upload_dir),
                version_notes="Explicit MOESCORE HF cache maintenance upload after verified rebuild.",
            )
        result.update(uploaded=True, uploaded_handle=dataset_handle, archive_sha256=manifest["sha256"])
    return result


def _utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


class RunSummary:
    """Atomic per-stage status ledger and final run metadata."""

    def __init__(
        self,
        path: str | os.PathLike[str],
        *,
        stage_names: Iterable[str],
        metadata: Mapping[str, Any] | None = None,
    ) -> None:
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.data: dict[str, Any] = {
            "schema_version": 1,
            "status": "RUNNING",
            "started_at_utc": _utc_now(),
            "completed_at_utc": None,
            "stages": {name: {"status": "PENDING"} for name in stage_names},
            "errors": [],
            "warnings": [],
            **dict(metadata or {}),
        }
        self._write()

    def _write(self) -> None:
        temporary = self.path.with_suffix(self.path.suffix + ".tmp")
        temporary.write_text(json.dumps(self.data, indent=2, sort_keys=True, default=str), encoding="utf-8")
        os.replace(temporary, self.path)

    def update_metadata(self, **values: Any) -> None:
        self.data.update(values)
        self._write()

    def warning(self, message: str) -> None:
        self.data.setdefault("warnings", []).append(str(message))
        self._write()

    def skip(self, name: str, reason: str) -> None:
        self.data["stages"][name] = {"status": "SKIPPED", "reason": reason, "completed_at_utc": _utc_now()}
        self._write()

    @contextmanager
    def stage(self, name: str):
        if name not in self.data["stages"]:
            self.data["stages"][name] = {"status": "PENDING"}
        self.data["stages"][name] = {"status": "RUNNING", "started_at_utc": _utc_now()}
        self._write()
        started = time.monotonic()
        try:
            yield self
        except BaseException as exc:
            detail = {"type": type(exc).__name__, "message": str(exc), "stage": name, "at_utc": _utc_now()}
            self.data["stages"][name] = {
                "status": "FAIL",
                "started_at_utc": self.data["stages"][name]["started_at_utc"],
                "completed_at_utc": _utc_now(),
                "duration_seconds": round(time.monotonic() - started, 3),
                "error": detail,
            }
            self.data.setdefault("errors", []).append(detail)
            self.data["status"] = "FAIL"
            self._write()
            raise
        else:
            current = self.data["stages"][name]
            current.update(
                status="PASS",
                completed_at_utc=_utc_now(),
                duration_seconds=round(time.monotonic() - started, 3),
            )
            self._write()

    def finish(self) -> dict[str, Any]:
        unfinished = [name for name, stage in self.data["stages"].items() if stage["status"] in {"PENDING", "RUNNING"}]
        if unfinished:
            raise RuntimeError(f"Cannot finish run with pending stages: {unfinished}")
        self.data["status"] = "FAIL" if self.data.get("errors") else "PASS"
        self.data["completed_at_utc"] = _utc_now()
        self._write()
        return self.data


__all__ = [
    "EXPERT10_CHECKPOINT_NAME",
    "EXPERT10_CHECKPOINT_SHA256",
    "HF_REQUIRED_MODELS",
    "RunSummary",
    "cache_shard_statistics",
    "ensure_source_checkout",
    "resolve_expert10_checkpoint",
    "resolve_hf_cache_inputs",
    "restore_hf_cache",
    "sha256_file",
    "validate_source_files",
]


## 01. Kiểm tra Python, CUDA và hai GPU

In [ ]:
import json, os, sys, torch

for path in (WORK_ROOT, CONFIG_ROOT, LOG_ROOT, RUNS_ROOT):
    path.mkdir(parents=True, exist_ok=True)
run_summary = RunSummary(
    WORK_ROOT / "run_summary.json",
    stage_names=RUN_STAGES,
    metadata={
        "expert_profile": EXPERT_PROFILE,
        "dataset_path": str(DATASET_ROOT),
        "configured_source_ref": REPO_REF,
        "configured_source_sha": REPO_COMMIT_SHA,
        "artifact_paths": {"root": str(WORK_ROOT), "configs": str(CONFIG_ROOT),
                           "logs": str(LOG_ROOT), "runs": str(RUNS_ROOT)},
    },
)
if SMOKE_ONLY:
    RUN_FULL_EXTRACTION = False
    RUN_GATE_TRAINING = False
    RUN_EVALUATION = False
    RUN_TEST_INFERENCE = False
    RUN_TEST_EVALUATION = False
    UPLOAD_FEATURE_CACHE = False
    run_summary.update_metadata(run_mode="SMOKE_ONLY")
with run_summary.stage("runtime_environment"):
    print("Python:", sys.version)
    print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available(), "device count:", torch.cuda.device_count())
    if sys.version_info[:2] not in {(3, 12), (3, 13)}:
        raise RuntimeError("Môi trường Kaggle cần Python 3.12 hoặc 3.13 theo requirements đã kiểm chứng.")
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError("Bật GPU accelerator hai GPU (T4 x2) trong Kaggle Notebook Settings.")
    GPU_NAMES = {f"cuda:{i}": torch.cuda.get_device_name(i) for i in (0, 1)}
    run_summary.update_metadata(python=sys.version, torch=torch.__version__,
                               cuda_runtime=torch.version.cuda, gpu_names=GPU_NAMES)
    print("GPU names:", GPU_NAMES)


## 02. Checkout source theo SHA chính xác

In [ ]:
import os, sys

with run_summary.stage("source_checkout"):
    actual_source_sha = ensure_source_checkout(REPO_URL, REPO_REF, REPO_COMMIT_SHA, PROJECT_ROOT)
    required_source_files = validate_source_files(PROJECT_ROOT)
    from model.expert_adapters.registry import EXPERT_PROFILES, profile_contract
    if tuple(EXPERT_PROFILES["full_10"]) != FULL10_ORDER:
        raise RuntimeError(f"Notebook Full10 order differs from registry: {EXPERT_PROFILES['full_10']}")
    order, dimensions, feature_width = profile_contract("full_10")
    if tuple(order) != FULL10_ORDER or len(order) != 10 or feature_width != 30472:
        raise RuntimeError(f"Sai Full10 profile contract: order={order}, width={feature_width}")
    if dimensions["m2d_lora_coral"] != 3073:
        raise RuntimeError("Expert 10 phải có feature width 3073.")
    if not (PROJECT_ROOT / "model/expert_adapters/m2d_lora_coral_adapter.py").is_file():
        raise FileNotFoundError("Pinned source thiếu M2DLoRACoralAdapter.")
    dirty = __import__("subprocess").run(
        ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError(f"Checkout source không sạch; không chạy source không xác định: {dirty[:500]}")
    sys.path.insert(0, str(PROJECT_ROOT))
    os.chdir(PROJECT_ROOT)
    print("Actual source SHA:", actual_source_sha)
    print("Required Full10 files:", len(required_source_files), "experts:", order, "width:", feature_width)
    run_summary.update_metadata(source_sha=actual_source_sha, expert_order=list(order),
                                feature_dimensions=dimensions, feature_width=feature_width)


## 03. Cài dependencies tương thích Python 3.12/3.13

In [ ]:
import importlib, importlib.metadata as metadata, subprocess
from scripts.kaggle_full9_dual_gpu import dependency_plan_for_python

with run_summary.stage("dependency_setup"):
    requirements_file, legacy_no_deps = dependency_plan_for_python(sys.version_info)
    print("Requirements:", requirements_file, "legacy --no-deps:", legacy_no_deps)
    loaded_before = {
        name: metadata.version(distribution)
        for name, distribution in (("numpy", "numpy"), ("scipy", "scipy"),
                                   ("transformers", "transformers"), ("librosa", "librosa"))
        if name in sys.modules
    }
    if INSTALL_MISSING_DEPENDENCIES:
        install_command = [
            sys.executable, str(PROJECT_ROOT / "scripts/install_kaggle_dependencies.py"),
            "--requirements", str(requirements_file),
            "--constraints", str(WORK_ROOT / "torch_constraints.txt"),
            "--report-json", str(LOG_ROOT / "dependencies_install.json"),
        ]
        if WHEELHOUSE:
            install_command += ["--wheelhouse", str(WHEELHOUSE)]
        subprocess.run(install_command, check=True, cwd=PROJECT_ROOT)
        if legacy_no_deps:
            legacy_command = [sys.executable, "-m", "pip", "install", "--no-deps"]
            if WHEELHOUSE:
                legacy_command += ["--no-index", "--find-links", str(WHEELHOUSE)]
            subprocess.run(legacy_command + ["msclap==1.3.4", "laion-clap==1.1.7"], check=True)
        importlib.invalidate_caches()
        changed_loaded = [
            name for name, old in loaded_before.items() if metadata.version(name) != old
        ]
        if changed_loaded:
            raise RuntimeError(
                "Dependency versions of already imported modules changed: " + ", ".join(changed_loaded)
                + ". Kernel restart required; Save & Run All must start from a fresh kernel."
            )
    imports = {
        "torch": "torch", "torchaudio": "torchaudio", "torchvision": "torchvision",
        "transformers": "transformers", "huggingface_hub": "huggingface-hub",
        "laion_clap": "laion-clap", "msclap": "msclap", "timm": "timm",
        "librosa": "librosa", "torchlibrosa": "torchlibrosa", "nnAudio": "nnAudio",
        "numba": "numba", "soundfile": "soundfile", "scipy": "scipy",
        "einops": "einops", "kagglehub": "kagglehub", "kaggle": "kaggle",
    }
    dependency_versions = {}
    for module, distribution in imports.items():
        try:
            importlib.import_module(module)
            dependency_versions[distribution] = metadata.version(distribution)
        except Exception as exc:
            raise RuntimeError(f"Dependency {distribution} import/runtime failure: {exc}") from exc
    if dependency_versions["msclap"] != "1.3.4" or dependency_versions["laion-clap"] != "1.1.7":
        raise RuntimeError("Dùng đúng msclap==1.3.4 và laion-clap==1.1.7 đã xác minh trong Full9.")
    if sys.version_info[:2] == (3, 13) and int(dependency_versions["numpy"].split(".")[0]) < 2:
        raise RuntimeError("Python 3.13 cần NumPy 2.x; không hạ xuống NumPy 1.x.")
    if dependency_versions["torch"] != torch.__version__ or not torch.cuda.is_available():
        raise RuntimeError("PyTorch/CUDA runtime đổi hoặc không khả dụng sau cài dependencies.")
    (LOG_ROOT / "dependency_versions.json").write_text(json.dumps(dependency_versions, indent=2))
    run_summary.update_metadata(dependency_versions=dependency_versions,
                                dependency_install_report=str(LOG_ROOT / "dependencies_install.json"))
    if legacy_no_deps:
        run_summary.warning("Python 3.13: metadata NumPy<2 legacy của MS-CLAP/LAION-CLAP được bỏ qua theo Full9; import/runtime preflight là điều kiện bắt buộc.")
    print(json.dumps(dependency_versions, indent=2))


## 04. Xác thực Kaggle Inputs và checkpoint Expert 10

In [ ]:
from scripts import kaggle_moe_pipeline as pipeline
from scripts.extract_moe_feature_store import FULL10_CHECKPOINT_FIELDS

with run_summary.stage("input_discovery"):
    print("Kaggle Inputs:", [str(p) for p in sorted(Path("/kaggle/input").iterdir())])
    if not DATASET_ROOT.is_dir():
        raise FileNotFoundError(f"Thiếu XACLE development dataset Input tại {DATASET_ROOT}.")
    for split in ("train", "validation", "test"):
        for expected_path in (DATASET_ROOT / f"meta_data/{split}_average.csv", DATASET_ROOT / f"wav/{split}"):
            if not expected_path.exists():
                raise FileNotFoundError(f"Dataset thiếu {split} resource: {expected_path}")
    if not WEIGHTS_ROOT.is_dir():
        raise FileNotFoundError(f"Thiếu weights dataset hoặc nested weights/weights: {WEIGHTS_ROOT}")
    expert10_path = resolve_expert10_checkpoint(
        "/kaggle/input", configured_path=EXPERT10_CHECKPOINT,
        expected_sha256=EXPERT10_EXPECTED_SHA256,
    )
    CHECKPOINT_OVERRIDES["m2d_lora_coral.finetuned_checkpoint"] = str(expert10_path)
    expert10_sha = sha256_file(expert10_path)
    run_summary.update_metadata(expert10_checkpoint=str(expert10_path),
                                expert10_checkpoint_sha256=expert10_sha)
    print("Dataset:", DATASET_ROOT)
    print("Weights root:", WEIGHTS_ROOT)
    print("Expert 10:", expert10_path, expert10_sha)


## 05. Khôi phục HF cache v2 đã sửa

In [ ]:
import os

with run_summary.stage("hf_cache_restore"):
    archive_path, manifest_path = resolve_hf_cache_inputs(
        "/kaggle/input", dataset_root=HF_CACHE_DATASET_ROOT,
        archive_path=HF_CACHE_ARCHIVE, manifest_path=HF_CACHE_MANIFEST,
    )
    hf_restore_report = restore_hf_cache(archive_path, manifest_path, HF_HOME)
    hub = HF_HOME / "hub"
    os.environ.update({
        "HF_HOME": str(HF_HOME), "HF_HUB_CACHE": str(hub), "TRANSFORMERS_CACHE": str(hub),
        "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1",
        "HF_HUB_DISABLE_TELEMETRY": "1", "TOKENIZERS_PARALLELISM": "false",
    })
    if hf_restore_report["status"] not in {"PASS", "REUSED_EXISTING"}:
        raise RuntimeError(hf_restore_report)
    run_summary.update_metadata(hf_cache={"dataset_root": str(HF_CACHE_DATASET_ROOT),
                                          "archive": str(archive_path),
                                          "manifest": str(manifest_path),
                                          "sha256": hf_restore_report["archive_sha256"],
                                          "hub": str(hub), "status": hf_restore_report["status"]})
    if UPLOAD_REPAIRED_HF_CACHE and not REBUILD_HF_CACHE:
        raise ValueError("UPLOAD_REPAIRED_HF_CACHE chỉ được bật cùng maintenance REBUILD_HF_CACHE.")
    if REBUILD_HF_CACHE:
        maintenance_report = rebuild_hf_cache(
            HF_HOME, upload=UPLOAD_REPAIRED_HF_CACHE,
            dataset_handle=HF_CACHE_UPLOAD_HANDLE, project_root=PROJECT_ROOT,
        )
        (LOG_ROOT / "hf_cache_maintenance.json").write_text(json.dumps(maintenance_report, indent=2))
    print("HF cache v2:", hf_restore_report)


## 06. Sinh cấu hình mười expert và cấu hình gate

In [ ]:
import json, yaml
from scripts.kaggle_full9_dual_gpu import validate_worker_devices
from scripts import kaggle_full9_artifacts as artifacts
from model.expert_adapters.registry import profile_contract

with run_summary.stage("config_generation"):
    order, feature_dimensions, total_width = profile_contract("full_10")
    if tuple(order) != FULL10_ORDER or len(order) != 10 or total_width != 30472:
        raise RuntimeError("Full10 registry order/dimension contract không khớp.")
    validate_worker_devices(WORKER_DEVICES, order)
    if set(EXTRACTION_BATCH_SIZES) != set(order) or any(v <= 0 for v in EXTRACTION_BATCH_SIZES.values()):
        raise ValueError("Cần đúng một extraction batch size dương cho mỗi expert.")
    defaults = {
        "expert1": {"pretrained_checkpoint": WEIGHTS_ROOT / "pretrained/laion-clap.pt", "finetuned_checkpoint": WEIGHTS_ROOT / "finetuned_weights/expert1_best.pt"},
        "expert2": {"pretrained_checkpoint": WEIGHTS_ROOT / "pretrained/mga-clap.pt", "finetuned_checkpoint": WEIGHTS_ROOT / "finetuned_weights/expert2_best.pt"},
        "expert3": {"base_checkpoint": WEIGHTS_ROOT / "pretrained/checkpoint-30.pth", "finetuned_checkpoint": WEIGHTS_ROOT / "finetuned_weights/expert3_best.pt"},
        "expert4": {"beats_checkpoint": WEIGHTS_ROOT / "pretrained/BEATs_iter3_plus_AS2M.pt", "clap_checkpoint": WEIGHTS_ROOT / "pretrained/laion-clap.pt", "finetuned_checkpoint": WEIGHTS_ROOT / "finetuned_weights/expert4_best.pt"},
        "top2_m2d": {"checkpoint": WEIGHTS_ROOT / "pretrained/checkpoint-30.pth"},
        "top2_msclap": {"checkpoint": WEIGHTS_ROOT / "pretrained/CLAP_weights_2023.pth"},
        "top2_laion": {"checkpoint": WEIGHTS_ROOT / "pretrained/laion-clap.pt"},
        "top2_mga": {"pretrained_checkpoint": WEIGHTS_ROOT / "pretrained/mga-clap.pt"},
        "m2d_lora": {"base_checkpoint": WEIGHTS_ROOT / "pretrained/checkpoint-30.pth", "finetuned_checkpoint": WEIGHTS_DATASET_ROOT / "m2d-lora.pt"},
        "m2d_lora_coral": {"base_checkpoint": WEIGHTS_ROOT / "pretrained/checkpoint-30.pth", "finetuned_checkpoint": expert10_path},
    }
    for dotted_key, override in CHECKPOINT_OVERRIDES.items():
        expert, field = dotted_key.split(".", 1)
        if expert not in defaults or field not in defaults[expert]:
            raise ValueError(f"Unknown checkpoint override {dotted_key!r}")
        defaults[expert][field] = Path(override)
    expert_checkpoints = {
        name: {field: str(Path(path).expanduser().resolve()) for field, path in fields.items()}
        for name, fields in defaults.items()
    }
    expert_config, gate_config, resolved = pipeline.generate_configs(
        PROJECT_ROOT, DATASET_ROOT, WEIGHTS_ROOT, HF_HOME, FEATURE_CACHE_ROOT,
        RUNS_ROOT, CONFIG_ROOT, expert_checkpoints=expert_checkpoints,
        profile="full_10", unilm_source_dir=UNILM_SOURCE_DIR,
    )
    gate_yaml = yaml.safe_load(gate_config.read_text(encoding="utf-8"))
    gate_yaml["runtime"].update(device=GATE_DEVICE, seed=SEED)
    gate_yaml["training"].update(batch_size=TRAIN_BATCH_SIZE, max_epochs=TRAIN_EPOCHS,
                                 learning_rate=TRAIN_LEARNING_RATE)
    gate_config.write_text(yaml.safe_dump(gate_yaml, sort_keys=False), encoding="utf-8")
    (CONFIG_ROOT / "worker_devices.json").write_text(json.dumps(WORKER_DEVICES, indent=2))
    from scripts.extract_moe_feature_store import checkpoint_provenance
    checkpoint_provenance_full10 = checkpoint_provenance(resolved, "full_10")
    run_summary.update_metadata(checkpoint_provenance=checkpoint_provenance_full10,
                                feature_dimensions=feature_dimensions,
                                feature_width=total_width,
                                config_paths={"expert": str(expert_config), "gate": str(gate_config),
                                              "worker_devices": str(CONFIG_ROOT / "worker_devices.json")})
    print("Order:", order, "width:", total_width)
    print("Expert config:", expert_config, "Gate config:", gate_config)


## 07. Offline preflight: checkpoints, dataset, text models và BART/LAION

In [ ]:
import subprocess, sys, json
from configs.moe_config import load_moe_config, resolve_moe_paths

with run_summary.stage("offline_preflight"):
    bart_preflight = "\n".join([
        "from transformers import BartTokenizer",
        "tok = BartTokenizer.from_pretrained('facebook/bart-base', local_files_only=True)",
        "assert tok.vocab_size > 0",
        "import laion_clap",
        "print('BART tokenizer and laion_clap import PASS')",
    ])
    isolated_env = os.environ.copy()
    subprocess.run([sys.executable, "-c", bart_preflight], env=isolated_env,
                   cwd=PROJECT_ROOT, check=True)
    resolved_config = resolve_moe_paths(load_moe_config(expert_config), expert_config)
    preflight_report = pipeline.preflight(
        resolved_config, DATASET_ROOT, WORK_ROOT,
        require_cuda=True, check_cache=True, allow_download=False,
    )
    if preflight_report.get("status") != "PASS":
        raise RuntimeError(f"Full10 offline preflight failed: {preflight_report}")
    (LOG_ROOT / "preflight.json").write_text(json.dumps(preflight_report, indent=2, default=str))
    # The five manifest entries are checked by hash before HF restore; native model IDs are checked offline here.
    print("Offline preflight PASS:", preflight_report["splits"], preflight_report["checkpoints"].keys())


## 08. Smoke extraction bốn mẫu trên hai GPU

In [ ]:
import json, subprocess, sys


def run_cli(log_path, *parts):
    command = [sys.executable, *map(str, parts)]
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("RUN", " ".join(command), flush=True)
    with log_path.open("w", encoding="utf-8") as log:
        result = subprocess.run(command, cwd=PROJECT_ROOT, stdout=log,
                                stderr=subprocess.STDOUT, check=False, text=True)
    if result.returncode:
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-100:]
        print("\n".join(tail))
        raise RuntimeError(f"Command failed ({result.returncode}); see {log_path}")
    return log_path

if RUN_GPU_SMOKE:
    with run_summary.stage("gpu_smoke"):
        smoke_report_path = LOG_ROOT / "smoke_extraction.json"
        run_cli(LOG_ROOT / "smoke_extraction.log",
                PROJECT_ROOT / "scripts/kaggle_full9_dual_gpu.py",
                "--config", expert_config, "--dataset-root", DATASET_ROOT,
                "--output-root", SMOKE_CACHE_ROOT,
                "--worker-devices-json", CONFIG_ROOT / "worker_devices.json",
                "--profile", EXPERT_PROFILE, "--smoke",
                "--batch-sizes", ",".join(str(EXTRACTION_BATCH_SIZES[name]) for name in order),
                "--shard-size", SHARD_SIZE, "--report-json", smoke_report_path)
        smoke_result = json.loads(smoke_report_path.read_text(encoding="utf-8"))
        if smoke_result.get("status") != "PASS":
            raise RuntimeError(smoke_result)
        smoke_validation = artifacts.validate_full10_store(
            smoke_result["feature_store_root"], {split: SMOKE_SAMPLES_PER_SPLIT for split in ("train", "validation", "test")}
        )
        run_summary.update_metadata(smoke_extraction={"report": str(smoke_report_path),
                                                        "feature_store": smoke_result["feature_store_root"],
                                                        "validation": smoke_validation})
        print("Smoke output:", smoke_validation)
else:
    run_summary.skip("gpu_smoke", "RUN_GPU_SMOKE=False")


## 09. Smoke test cho gate

In [ ]:
if RUN_GPU_SMOKE:
    with run_summary.stage("smoke_gate"):
        smoke_gate_report = pipeline.smoke_store(smoke_result["feature_store_root"], gate_yaml)
        if smoke_gate_report.get("status") != "PASS" or smoke_gate_report.get("optimizer_steps") != 3:
            raise RuntimeError(smoke_gate_report)
        (LOG_ROOT / "smoke_gate.json").write_text(json.dumps(smoke_gate_report, indent=2))
        run_summary.update_metadata(smoke_gate={"status": smoke_gate_report["status"],
                                               "optimizer_steps": smoke_gate_report["optimizer_steps"]})
        print("Smoke gate PASS; optimizer steps:", smoke_gate_report["optimizer_steps"])
else:
    run_summary.skip("smoke_gate", "RUN_GPU_SMOKE=False")


## 10. Khôi phục cache Full9 tùy chọn trước Full10 extraction

In [ ]:
import shutil

RESTORE_EXISTING_FEATURE_CACHE = EXISTING_FEATURE_CACHE

def copy_cache_directory_without_symlinks(source, destination):
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite existing feature cache: {destination}")
    if not source.is_dir():
        raise NotADirectoryError(source)
    for item in source.rglob("*"):
        if item.is_symlink():
            raise ValueError(f"Feature cache directory contains a symlink: {item}")
    shutil.copytree(source, destination, symlinks=False)

if EXISTING_FEATURE_CACHE is None:
    run_summary.skip("existing_feature_cache_restore", "EXISTING_FEATURE_CACHE is None; extraction will use any local partial shards or start fresh")
else:
    with run_summary.stage("existing_feature_cache_restore"):
        mounted = Path(EXISTING_FEATURE_CACHE).expanduser().resolve()
        cache_dir = FEATURE_CACHE_ROOT / "feature_cache"
        if not cache_dir.exists():
            FEATURE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
            if (mounted / "upload_manifest.json").is_file():
                restore_manifest = artifacts.restore_feature_cache(mounted, FEATURE_CACHE_ROOT)
                restore_mode = "verified feature_cache.tar and upload_manifest.json"
            elif (mounted / "feature_cache").is_dir():
                copy_cache_directory_without_symlinks(mounted / "feature_cache", cache_dir)
                restore_manifest = {"mode": "directory input; shard sidecar/hash validation is deferred to workers"}
                restore_mode = "directory with worker-level per-shard validation"
            elif mounted.name == "feature_cache" and mounted.is_dir():
                copy_cache_directory_without_symlinks(mounted, cache_dir)
                restore_manifest = {"mode": "directory input; shard sidecar/hash validation is deferred to workers"}
                restore_mode = "directory with worker-level per-shard validation"
            else:
                raise FileNotFoundError("Cache Input needs upload_manifest.json + feature_cache.tar or feature_cache/ directory.")
        else:
            restore_manifest = {"mode": "existing local cache; preserved and revalidated by workers"}
            restore_mode = restore_manifest["mode"]
        run_summary.update_metadata(existing_cache={"input": str(mounted), "mode": restore_mode,
                                                    "restore_manifest": restore_manifest})
        print("Existing cache restored before extraction:", restore_mode, restore_manifest)


## 11. Full10 extraction/resume hai GPU

In [ ]:
if RUN_FULL_EXTRACTION and not SMOKE_ONLY:
    with run_summary.stage("full_extraction"):
        if EXPERT_PROFILE != "full_10":
            raise RuntimeError("Full extraction must use profile=full_10")
        full_report_path = LOG_ROOT / "full_extraction.json"
        # Existing script calls prepare_job(profile="full_10"), compares cached identities,
        # validates shards and completed_stores_match before launching workers.
        run_cli(LOG_ROOT / "full_extraction.log",
                PROJECT_ROOT / "scripts/kaggle_full9_dual_gpu.py",
                "--config", expert_config, "--dataset-root", DATASET_ROOT,
                "--output-root", FEATURE_CACHE_ROOT,
                "--worker-devices-json", CONFIG_ROOT / "worker_devices.json",
                "--profile", "full_10",
                "--batch-sizes", ",".join(str(EXTRACTION_BATCH_SIZES[name]) for name in FULL10_ORDER),
                "--shard-size", SHARD_SIZE, "--report-json", full_report_path)
        full_result = json.loads(full_report_path.read_text(encoding="utf-8"))
        if full_result.get("status") != "PASS":
            raise RuntimeError(full_result)
        if not Path(full_result["feature_store_root"]).is_dir():
            raise FileNotFoundError(full_result["feature_store_root"])
        run_summary.update_metadata(full_extraction={"status": full_result["status"],
            "feature_store_root": full_result["feature_store_root"],
            "assembly_id": full_result.get("assembly_id"),
            "cache_reuse": full_result.get("cache_reuse", {}),
            "cache_identity_audit": full_result.get("cache_identity_audit", {}),
            "workers": full_result.get("workers", {})})
        print("Full10 extraction PASS:", full_result["feature_store_root"])
        print("Validated cache reuse statistics:", full_result.get("cache_reuse", "see worker runtime reports"))
else:
    if SMOKE_ONLY:
        run_summary.skip("full_extraction", "SMOKE_ONLY=True")
    else:
        run_summary.skip("full_extraction", "RUN_FULL_EXTRACTION=False; kiểm tra Full10 store có sẵn bằng identity đầy đủ")
        from scripts.extract_moe_feature_store import prepare_job, completed_stores_match, assembly_store_root, job_identity
        _, expected_job, expected_manifests = prepare_job(
            expert_config, DATASET_ROOT, FEATURE_CACHE_ROOT,
            batch_sizes=EXTRACTION_BATCH_SIZES, shard_size=SHARD_SIZE,
            profile="full_10", worker_devices=WORKER_DEVICES,
        )
        if not completed_stores_match(expected_job, expected_manifests, FEATURE_CACHE_ROOT):
            raise FileNotFoundError("Không có Full10 feature store hoàn chỉnh khớp provenance; bật RUN_FULL_EXTRACTION=True.")
        full_result = {"status": "PASS", "reused_complete_store": True,
                       "feature_store_root": str(assembly_store_root(FEATURE_CACHE_ROOT, expected_job)),
                       "assembly_id": job_identity(expected_job), "profile": "full_10"}
        (LOG_ROOT / "full_extraction.json").write_text(json.dumps(full_result, indent=2))


## 12. Xác thực feature store Full10

In [ ]:
import json, yaml
from data.moe_feature_store import validate_feature_store

if SMOKE_ONLY:
    run_summary.skip("feature_store_validation", "SMOKE_ONLY=True; only the four-row smoke store was validated")
else:
    with run_summary.stage("feature_store_validation"):
        full_store = Path(full_result["feature_store_root"])
        store_validation = artifacts.validate_full10_store(
            full_store, {"train": 7500, "validation": 3000, "test": 3000}
        )
        detailed = {}
        for split in ("train", "validation", "test"):
            manifest = validate_feature_store(full_store, split=split, require_labels=split != "test")
            if manifest["feature_width"] != 30472 or len(manifest["expert_order"]) != 10:
                raise ValueError(f"{split}: wrong Full10 shape contract")
            if split == "test" and manifest.get("labels_present"):
                raise ValueError("Test feature store must be label-free.")
            detailed[split] = {"samples": sum(shard["count"] for shard in manifest["shards"]),
                               "scores_shape": [sum(shard["count"] for shard in manifest["shards"]), 10],
                               "features_shape": [sum(shard["count"] for shard in manifest["shards"]), 30472],
                               "store_identity": manifest["store_identity_sha256"],
                               "source_manifest_identity": manifest["source_manifest_sha256"],
                               "labels_present": manifest["labels_present"]}
        gate_yaml["data"]["feature_store_root"] = str(full_store)
        gate_config.write_text(yaml.safe_dump(gate_yaml, sort_keys=False), encoding="utf-8")
        (LOG_ROOT / "full_store_validation.json").write_text(json.dumps({"status": "PASS", "splits": detailed,
                                                                            "cache_validation": store_validation}, indent=2))
        run_summary.update_metadata(feature_store_identity={split: values["store_identity"] for split, values in detailed.items()},
                                    source_manifest_identity={split: values["source_manifest_identity"] for split, values in detailed.items()},
                                    feature_store_root=str(full_store))
        print(json.dumps(detailed, indent=2))


## 13. Đóng gói cache riêng tư (tắt mặc định)

In [ ]:
if not SMOKE_ONLY and UPLOAD_FEATURE_CACHE:
    with run_summary.stage("feature_cache_upload"):
        package = artifacts.package_feature_cache(FEATURE_CACHE_ROOT, UPLOAD_DIRECTORY, stores=store_validation)
        upload_report = artifacts.upload_feature_cache(
            FEATURE_DATASET_HANDLE, UPLOAD_DIRECTORY,
            version_notes=f"MOESCORE full_10 {full_result['assembly_id']}",
        )
        (LOG_ROOT / "feature_cache_upload.json").write_text(json.dumps(upload_report, indent=2))
        run_summary.update_metadata(feature_cache_upload=upload_report)
        print("Private cache upload:", upload_report)
else:
    run_summary.skip("feature_cache_upload", "SMOKE_ONLY=True or UPLOAD_FEATURE_CACHE=False; no Kaggle upload is performed")


## 14. Train/resume gate Full10

In [ ]:
import json, subprocess

if RUN_GATE_TRAINING and not SMOKE_ONLY:
    with run_summary.stage("gate_training"):
        best_path, latest_path = RUNS_ROOT / "best_gate.pt", RUNS_ROOT / "latest_gate.pt"
        existing = [str(path) for path in (best_path, latest_path) if path.exists()]
        if GATE_RESUME_CHECKPOINT is None and existing:
            raise FileExistsError(f"Gate outputs already exist {existing}; choose a fresh RUNS_ROOT or an explicit compatible Full10 resume checkpoint.")
        training_args = [PROJECT_ROOT / "scripts/train_moe_gating.py", "--config", gate_config,
                         "--device", GATE_DEVICE, "--report-json", LOG_ROOT / "training.json"]
        if GATE_RESUME_CHECKPOINT is not None:
            resume_path = Path(GATE_RESUME_CHECKPOINT).expanduser().resolve()
            if not resume_path.is_file():
                raise FileNotFoundError(f"Full10 resume checkpoint does not exist: {resume_path}")
            training_args += ["--resume", resume_path]
        run_cli(LOG_ROOT / "gate_training.log", *training_args)
        training_report = json.loads((LOG_ROOT / "training.json").read_text(encoding="utf-8"))
        if training_report.get("status") != "PASS" or training_report.get("expert_model_construction") is not False:
            raise RuntimeError(f"Gating training did not pass/froze expert models: {training_report}")
        if not best_path.is_file() or not latest_path.is_file():
            raise FileNotFoundError("Training report passed without both best_gate.pt and latest_gate.pt.")
        history_path = pipeline.record_training(LOG_ROOT / "training.json", RUNS_ROOT)
        best_epoch = max(training_report.get("history", []),
                         key=lambda row: row["validation_metrics_raw_scale"]["srcc"])["epoch"] if training_report.get("history") else None
        run_summary.update_metadata(training={"status": "PASS", "best_checkpoint": str(best_path),
            "latest_checkpoint": str(latest_path), "best_srcc": training_report.get("best_srcc_raw_scale"),
            "epochs_completed": training_report.get("epochs_completed"), "best_epoch_zero_based": best_epoch,
            "global_step": training_report.get("global_step"), "history": str(history_path),
            "resume_checkpoint": str(GATE_RESUME_CHECKPOINT) if GATE_RESUME_CHECKPOINT else None})
        if best_epoch is not None:
            run_summary.update_metadata(best_epoch_zero_based=best_epoch)
        print("Training PASS:", training_report.get("epochs_completed"), "epochs; best SRCC:", training_report.get("best_srcc_raw_scale"))
else:
    run_summary.skip("gate_training", "SMOKE_ONLY=True or RUN_GATE_TRAINING=False")


## 15. Đánh giá validation, chọn checkpoint và xuất dự đoán có identity

In [ ]:
import csv, json, math
from torch.utils.data import DataLoader
from data.moe_training_dataset import MoeTrainingDataset, collate_moe_features
from data.moe_feature_store import preprocessing_contract
from model.moe_gating_core import GatingNetwork
from training.checkpoint_manager import MoeCheckpointManager
from training.moe_evaluator import MoeGateEvaluator

if RUN_EVALUATION and not SMOKE_ONLY:
    with run_summary.stage("validation_evaluation"):
        best = RUNS_ROOT / "best_gate.pt"
        if not best.is_file():
            raise FileNotFoundError(f"Full10 best gate checkpoint is missing: {best}")
        validation_metrics_path = RUNS_ROOT / "validation_metrics.json"
        run_cli(LOG_ROOT / "validation_evaluation.log",
                PROJECT_ROOT / "scripts/evaluate_moe_gating.py", "--config", gate_config,
                "--checkpoint", best, "--device", GATE_DEVICE,
                "--report-json", validation_metrics_path)
        metrics_report = json.loads(validation_metrics_path.read_text(encoding="utf-8"))
        if metrics_report.get("status") != "PASS" or metrics_report.get("sample_count") != 3000:
            raise RuntimeError(f"Validation must evaluate all 3000 rows: {metrics_report}")
        raw_metrics = metrics_report["metrics_raw_scale"]
        for metric in ("srcc", "lcc", "ktau", "mse", "mae"):
            if metric not in raw_metrics:
                raise ValueError(f"Validation metrics missing {metric}")
        validation = MoeTrainingDataset(str(full_store), split="validation", require_labels=True)
        order, _, width = profile_contract("full_10")
        gate = GatingNetwork(input_feature_dim=width, num_experts=len(order), profile="full_10").to(GATE_DEVICE)
        gate_payload = MoeCheckpointManager().load_for_inference(
            best, gating_network=gate,
            expected_expert_checkpoint_provenance=validation.manifest["checkpoint_provenance"],
        )
        if gate_payload["feature_store_identity"]["validation"] != validation.identity:
            raise ValueError("Validation sample/feature-store identity differs from selected gate checkpoint.")
        if gate_payload["feature_store_identity"]["preprocessing_contract"] != preprocessing_contract(validation.manifest):
            raise ValueError("Validation preprocessing identity differs from selected Full10 checkpoint.")
        loader = DataLoader(validation, batch_size=gate_yaml["validation"]["batch_size"],
                            shuffle=False, num_workers=0, collate_fn=collate_moe_features)
        prediction_result = MoeGateEvaluator(gate, gate_yaml["training"]["loss"]).evaluate(loader)
        validation_source = pipeline.verified_source_manifest(full_store, "validation")["records"]
        expected_ids = [row["sample_id"] for row in validation_source]
        observed_pairs = list(zip(prediction_result["sample_ids"], prediction_result["wave_file_names"]))
        expected_pairs = [(row["sample_id"], row["wave_file_name"]) for row in validation_source]
        if [row[0] for row in observed_pairs] != expected_ids or observed_pairs != expected_pairs:
            raise ValueError("Validation predictions changed source sample ordering.")
        csv_path = RUNS_ROOT / "validation_predictions.csv"
        with csv_path.open("w", encoding="utf-8", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=["sample_id", "wave_file_name", "pred_score"])
            writer.writeheader()
            writer.writerows({"sample_id": sid, "wave_file_name": name, "pred_score": float(score)}
                             for sid, name, score in zip(prediction_result["sample_ids"],
                                                         prediction_result["wave_file_names"],
                                                         prediction_result["prediction_raw"].tolist()))
        with csv_path.open(encoding="utf-8") as stream:
            output_line_count = sum(1 for _ in stream)
        if output_line_count != 3001:
            raise ValueError("Validation prediction CSV row count is not 3000 plus header.")
        run_summary.update_metadata(best_epoch_zero_based=gate_payload["epoch"])
        run_summary.update_metadata(validation={"status": "PASS", "sample_count": 3000,
            "checkpoint": str(best), "checkpoint_epoch": gate_payload["epoch"],
            "metrics": raw_metrics, "predictions": str(csv_path), "metrics_path": str(validation_metrics_path)})
        print("Validation metrics:", {"SRCC": raw_metrics["srcc"], "LCC": raw_metrics["lcc"],
                                       "KTAU": raw_metrics["ktau"], "MSE": raw_metrics["mse"],
                                       "MAE": raw_metrics["mae"], "n": 3000})
else:
    run_summary.skip("validation_evaluation", "SMOKE_ONLY=True or RUN_EVALUATION=False")


## 16. Label-free test inference và kiểm tra submission order

In [ ]:
import csv, json, math, shutil

if RUN_TEST_INFERENCE and not SMOKE_ONLY:
    with run_summary.stage("test_inference"):
        best = RUNS_ROOT / "best_gate.pt"
        if not best.is_file():
            raise FileNotFoundError(f"Full10 checkpoint required for test inference: {best}")
        submission_path = RUNS_ROOT / "submission.csv"
        infer_report_path = RUNS_ROOT / "test_inference.json"
        run_cli(LOG_ROOT / "test_inference.log",
                PROJECT_ROOT / "scripts/infer_moe_gating.py", "--config", gate_config,
                "--checkpoint", best, "--device", GATE_DEVICE,
                "--output", submission_path, "--report-json", infer_report_path)
        inference_report = json.loads(infer_report_path.read_text(encoding="utf-8"))
        if inference_report.get("status") != "PASS" or inference_report.get("labels_required") is not False:
            raise RuntimeError(f"Test inference must be label-free: {inference_report}")
        if inference_report.get("sample_count") != 3000:
            raise ValueError(f"Test inference expected 3000 rows: {inference_report}")
        source = pipeline.verified_source_manifest(full_store, "test")
        if any("label_raw" in row for row in source["records"]):
            raise ValueError("Test source manifest unexpectedly contains labels.")
        with submission_path.open(encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != ["wav_file_name", "pred_score"]:
                raise ValueError(f"Unexpected submission schema: {reader.fieldnames}")
            submission_rows = list(reader)
        if len(submission_rows) != 3000:
            raise ValueError("submission.csv must contain 3000 predictions.")
        expected_names = [row["wave_file_name"] for row in source["records"]]
        observed_names = [row["wav_file_name"] for row in submission_rows]
        scores = [float(row["pred_score"]) for row in submission_rows]
        if observed_names != expected_names or len(set(observed_names)) != 3000:
            raise ValueError("Submission sample identity/order differs from the test feature store.")
        if not all(math.isfinite(value) for value in scores):
            raise ValueError("Submission has non-finite scores.")
        enriched_path = RUNS_ROOT / "test_predictions.csv"
        with enriched_path.open("w", encoding="utf-8", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=["sample_id", "wave_file_name", "pred_score"])
            writer.writeheader()
            writer.writerows({"sample_id": record["sample_id"], "wave_file_name": record["wave_file_name"],
                              "pred_score": score}
                             for record, score in zip(source["records"], scores))
        run_summary.update_metadata(test_inference={"status": "PASS", "sample_count": 3000,
            "labels_required": False, "submission": str(submission_path),
            "predictions": str(enriched_path), "report": str(infer_report_path),
            "sample_order_verified": True})
        print("Label-free test inference PASS; submission rows:", len(submission_rows))
else:
    run_summary.skip("test_inference", "SMOKE_ONLY=True or RUN_TEST_INFERENCE=False")


## 17. Đánh giá test tùy chọn, chỉ sau khi chọn model bằng validation

In [ ]:
if RUN_TEST_EVALUATION and not SMOKE_ONLY:
    if not RUN_EVALUATION:
        raise ValueError("Test evaluation yêu cầu validation checkpoint selection đã chạy trong cùng lượt.")
    with run_summary.stage("test_evaluation"):
        posthoc = pipeline.evaluate_best(gate_yaml, DATASET_ROOT, checkpoint=RUNS_ROOT / "best_gate.pt")
        if posthoc["test"]["sample_count"] != 3000:
            raise ValueError(f"Optional development-test evaluation row count mismatch: {posthoc['test']}")
        (RUNS_ROOT / "metrics.json").write_text(json.dumps(posthoc, indent=2, default=str))
        # evaluate_best is a post-hoc metric stage and writes its own CSV; restore the
        # label-free identity-rich test_predictions.csv artifact after its label join.
        test_source_records = pipeline.verified_source_manifest(full_store, "test")["records"]
        with (RUNS_ROOT / "test_predictions.csv").open(encoding="utf-8", newline="") as stream:
            prediction_reader = csv.DictReader(stream)
            evaluated_rows = list(prediction_reader)
        if [row.get("wav_file_name") for row in evaluated_rows] != [row["wave_file_name"] for row in test_source_records]:
            raise ValueError("Post-hoc test evaluation changed the original test sample order.")
        with (RUNS_ROOT / "test_predictions.csv").open("w", encoding="utf-8", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=["sample_id", "wave_file_name", "pred_score"])
            writer.writeheader()
            writer.writerows({"sample_id": source["sample_id"], "wave_file_name": source["wave_file_name"],
                              "pred_score": float(row["pred_score"])}
                             for source, row in zip(test_source_records, evaluated_rows))
        run_summary.update_metadata(optional_test_evaluation={"status": "PASS",
            "metrics": posthoc["test"], "metrics_path": str(RUNS_ROOT / "test_metrics.json")})
        print("Optional post-hoc development-test metrics:", posthoc["test"])
else:
    run_summary.skip("test_evaluation", "SMOKE_ONLY=True or RUN_TEST_EVALUATION=False; no test labels are read by this notebook stage")


## 18. Xác nhận artifacts và run_summary cuối

In [ ]:
import json

with run_summary.stage("final_artifact_verification"):
    required_artifacts = {
        "run_summary": WORK_ROOT / "run_summary.json",
        "expert_config": CONFIG_ROOT / "moe_expert_config.yaml",
        "gate_config": CONFIG_ROOT / "moe_gating_config.yaml",
        "worker_devices": CONFIG_ROOT / "worker_devices.json",
        "preflight": LOG_ROOT / "preflight.json",
    }
    if RUN_GPU_SMOKE:
        required_artifacts["smoke_extraction"] = LOG_ROOT / "smoke_extraction.json"
        required_artifacts["smoke_gate"] = LOG_ROOT / "smoke_gate.json"
    if not SMOKE_ONLY:
        required_artifacts["full_extraction"] = LOG_ROOT / "full_extraction.json"
        required_artifacts["feature_store_validation"] = LOG_ROOT / "full_store_validation.json"
    if RUN_GATE_TRAINING and not SMOKE_ONLY:
        required_artifacts["training"] = LOG_ROOT / "training.json"
        required_artifacts["training_history"] = RUNS_ROOT / "training_history.json"
        required_artifacts["best_gate"] = RUNS_ROOT / "best_gate.pt"
        required_artifacts["latest_gate"] = RUNS_ROOT / "latest_gate.pt"
    if (RUN_EVALUATION or RUN_TEST_INFERENCE) and not SMOKE_ONLY:
        required_artifacts["best_gate"] = RUNS_ROOT / "best_gate.pt"
    if RUN_EVALUATION and not SMOKE_ONLY:
        required_artifacts["validation_metrics"] = RUNS_ROOT / "validation_metrics.json"
        required_artifacts["validation_predictions"] = RUNS_ROOT / "validation_predictions.csv"
    if RUN_TEST_INFERENCE and not SMOKE_ONLY:
        required_artifacts["submission"] = RUNS_ROOT / "submission.csv"
        required_artifacts["test_predictions"] = RUNS_ROOT / "test_predictions.csv"
        required_artifacts["test_inference"] = RUNS_ROOT / "test_inference.json"
    if RUN_TEST_EVALUATION and not SMOKE_ONLY:
        required_artifacts["test_metrics"] = RUNS_ROOT / "test_metrics.json"
        required_artifacts["combined_metrics"] = RUNS_ROOT / "metrics.json"
        required_artifacts["test_predictions"] = RUNS_ROOT / "test_predictions.csv"
    missing_artifacts = [name for name, path in required_artifacts.items() if not path.is_file()]
    if missing_artifacts:
        raise FileNotFoundError(f"Required artifacts were not produced: {missing_artifacts}")
    if not SMOKE_ONLY and not (FEATURE_CACHE_ROOT / "feature_cache").is_dir():
        raise FileNotFoundError("Per-expert feature cache directory is missing.")
    run_summary.update_metadata(artifact_paths={name: str(path) for name, path in required_artifacts.items()},
                                run_status="artifacts_verified")
final_summary = run_summary.finish()
print("Full10 run status:", final_summary["status"])
print("Summary:", WORK_ROOT / "run_summary.json")
print("Artifacts:", json.dumps({name: str(path) for name, path in required_artifacts.items()}, indent=2))
